# HeatShield personal-risk proposal charts

These reproducible charts exercise the current explainable `heatcore.risk.assess_risk` pilot model. Weather and scenarios are illustrative, not observed Delhi data or calibrated health predictions. The bands are screening proposals, not regulatory exposure limits.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_candidates = (Path.cwd(), *Path.cwd().parents)
REPO_ROOT = next(
    (path for path in repo_candidates if (path / "heatcore" / "risk.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository or one of its subfolders")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from heatcore.risk import (
    EXTREME_SCORE_THRESHOLD,
    HIGH_SCORE_THRESHOLD,
    MODERATE_SCORE_THRESHOLD,
    assess_risk,
)

FIGURE_DIR = REPO_ROOT / "docs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PILOT_LOCATION = {"lat": 28.75, "lon": 77.12}


def score_scenario(
    *,
    activity: str,
    duration_min: int,
    profile: str,
    weather: dict[str, float],
) -> dict[str, object]:
    """Return the risk API response for one illustrative scenario."""
    return assess_risk(
        **PILOT_LOCATION,
        activity=activity,
        duration_min=duration_min,
        profile=profile,
        weather=weather,
    )


## 1. Proposal: risk across a representative Delhi day

Synthetic hourly air temperature, relative humidity, wind, and solar load are passed through the same WBGT and risk functions used by the API.

In [ ]:
hours = np.arange(6, 21)
daylight_fraction = np.sin(np.pi * (hours - 6) / 14)
air_temps_c = 26 + 13 * daylight_fraction
solar_load_w_m2 = 900 * daylight_fraction
relative_humidity_pct = 68 - 28 * daylight_fraction
risk_scores = []

for hour, temp_c, humidity_pct, solar_w_m2 in zip(
    hours,
    air_temps_c,
    relative_humidity_pct,
    solar_load_w_m2,
):
    result = score_scenario(
        activity="walking",
        duration_min=60,
        profile="general",
        weather={
            "temp_c": float(temp_c),
            "relative_humidity_pct": float(humidity_pct),
            "wind_speed_m_s": 1.2,
            "solar_radiation_w_m2": float(solar_w_m2),
        },
    )
    risk_scores.append(result["risk_score"])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(hours, risk_scores, color="#c2410c", marker="o", linewidth=2.5)
ax.fill_between(hours, risk_scores, color="#fb923c", alpha=0.2)
for threshold, label in (
    (MODERATE_SCORE_THRESHOLD, "Moderate"),
    (HIGH_SCORE_THRESHOLD, "High"),
    (EXTREME_SCORE_THRESHOLD, "Extreme"),
):
    ax.axhline(threshold, color="#64748b", linestyle="--", linewidth=0.9)
    ax.text(20.1, threshold, label, va="center", fontsize=8, color="#475569")
ax.set(title="Proposal 1 — Personal risk by time of day", xlabel="Local hour", ylabel="Risk score (0–100)")
ax.set(xticks=hours, xlim=(6, 20.8), ylim=(0, 100))
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "risk_by_time_of_day.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print("Exported risk_by_time_of_day.png")


## 2. Proposal: same 40°C air temperature, different people and environments

Both scenarios hold air temperature and exposure duration constant. Humidity, wind, solar exposure, activity, and vulnerability profile vary, illustrating why air temperature alone is not a personal-risk estimate.

In [ ]:
scenarios = [
    {
        "label": "General profile\nwalking, shaded/breezy",
        "activity": "walking",
        "profile": "general",
        "weather": {
            "temp_c": 40.0,
            "relative_humidity_pct": 35.0,
            "wind_speed_m_s": 2.5,
            "solar_radiation_w_m2": 180.0,
        },
    },
    {
        "label": "Older adult\noutdoor work, sunny/still",
        "activity": "outdoor_work",
        "profile": "older_adult",
        "weather": {
            "temp_c": 40.0,
            "relative_humidity_pct": 60.0,
            "wind_speed_m_s": 0.4,
            "solar_radiation_w_m2": 950.0,
        },
    },
]
scenario_results = [
    score_scenario(
        activity=scenario["activity"],
        duration_min=60,
        profile=scenario["profile"],
        weather=scenario["weather"],
    )
    for scenario in scenarios
]
scenario_scores = [result["risk_score"] for result in scenario_results]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(
    [scenario["label"] for scenario in scenarios],
    scenario_scores,
    color=["#0f766e", "#b91c1c"],
    width=0.58,
)
for bar, result in zip(bars, scenario_results):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        min(result["risk_score"] + 2, 108),
        f"{result['risk_score']:.0f} — {result['level']}\nWBGT {result['wbgt_c']:.1f}°C",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set(title="Proposal 2 — Same 40°C air temperature, different exposures", ylabel="Risk score (0–100)")
ax.set_ylim(0, 112)
ax.set_yticks(np.arange(0, 101, 20))
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "same_40c_day_comparison.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print("Exported same_40c_day_comparison.png")


## 3. Proposal: one-factor-at-a-time sensitivity

Each range changes one factor from a 30°C WBGT, one-hour walking, general-profile baseline. The plotted score deltas make the model's assumed sensitivity explicit; they are not uncertainty intervals.

In [ ]:
baseline = {
    "wbgt_c": 30.0,
    "activity": "walking",
    "duration_min": 60,
    "profile": "general",
}


def score_from_inputs(**changes: object) -> float:
    """Score a sensitivity case while holding unspecified inputs at baseline."""
    inputs = {**baseline, **changes}
    result = score_scenario(
        activity=inputs["activity"],
        duration_min=inputs["duration_min"],
        profile=inputs["profile"],
        weather={"wbgt_c": inputs["wbgt_c"]},
    )
    return result["risk_score"]


baseline_score = score_from_inputs()
sensitivity_ranges = [
    ("WBGT (°C)", {"wbgt_c": 27.0}, {"wbgt_c": 33.0}),
    ("Duration (min)", {"duration_min": 30}, {"duration_min": 120}),
    ("Activity", {"activity": "resting"}, {"activity": "running"}),
    ("Profile", {"profile": "general"}, {"profile": "chronic_condition"}),
]
deltas = [
    (
        score_from_inputs(**lower) - baseline_score,
        score_from_inputs(**upper) - baseline_score,
    )
    for _, lower, upper in sensitivity_ranges
]

fig, ax = plt.subplots(figsize=(8, 4.5))
for index, ((lower_delta, upper_delta), (label, _, _)) in enumerate(
    zip(deltas, sensitivity_ranges)
):
    left = min(lower_delta, upper_delta)
    width = abs(upper_delta - lower_delta)
    ax.barh(index, width, left=left, color="#2563eb", height=0.48, alpha=0.8)
    ax.plot([lower_delta, upper_delta], [index, index], color="#1e3a8a", marker="|", linewidth=1.5)
    ax.text(lower_delta, index - 0.18, f"{lower_delta:+.0f}", ha="center", va="bottom", fontsize=8)
    ax.text(upper_delta, index - 0.18, f"{upper_delta:+.0f}", ha="center", va="bottom", fontsize=8)
ax.axvline(0, color="#334155", linewidth=1)
ax.set_yticks(range(len(sensitivity_ranges)), [item[0] for item in sensitivity_ranges])
ax.invert_yaxis()
ax.set(
    title=f"Proposal 3 — Sensitivity around baseline score {baseline_score:.0f}",
    xlabel="Risk-score change from baseline (points)",
)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "risk_sensitivity.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print("Exported risk_sensitivity.png")
